In [4]:
import pandas as pd
import numpy as np
import pyfixest as pf

from pathlib import Path

In [5]:
ROOT = Path.cwd()
covid_df = pd.read_csv( ROOT / 'school_covid.csv')
covid_df.dtypes

district_id           int64
enrolment           float64
pass_percent        float64
case_rate           float64
inperson_percent    float64
virtual_percent     float64
hybrid_percent      float64
year                  int64
pass8               float64
participation       float64
missing_lunch         int64
share_black         float64
share_white         float64
share_lunch         float64
share_ell           float64
missing_ell           int64
unemployment        float64
dtype: object

In [7]:
covid_df.isna().sum()

district_id           0
enrolment             0
pass_percent          0
case_rate             0
inperson_percent      0
virtual_percent     803
hybrid_percent        0
year                  0
pass8               325
participation         0
missing_lunch         0
share_black           0
share_white           0
share_lunch         343
share_ell           284
missing_ell           0
unemployment          0
dtype: int64

In [8]:
covid_df.describe()


,district_id,enrolment,pass_percent,case_rate,inperson_percent,virtual_percent,hybrid_percent,year,pass8,participation,missing_lunch,share_black,share_white,share_lunch,share_ell,missing_ell,unemployment
count,1.519000e+03,1519.000000,1519.000000,1519.000000,1519.000000,716.000000,1519.000000,1519.0,1194.000000,1519.000000,1519.000000,1519.000000,1519.000000,1176.000000,1235.000000,1519.000000,1519.000000
mean,3.752074e+09,26.773364,46.482433,26.483867,48.708037,23.466408,40.230773,2021.0,0.424434,92.539800,0.225806,0.100202,0.737696,0.450882,0.037294,0.186965,6.479641
std,3.010687e+10,37.260312,18.660544,6.320168,36.927684,22.698184,34.081319,0.0,0.205247,11.152634,0.418250,0.203173,0.243198,0.234006,0.056076,0.390012,1.650059
min,1.000000e+00,0.400000,0.000000,9.507864,0.000000,0.349000,0.000000,2021.0,0.000000,1.415094,0.000000,0.000000,0.000000,0.003023,0.000723,0.000000,2.508333
25%,6.910000e+03,7.505000,32.766665,22.943220,20.000000,7.664000,2.555000,2021.0,0.261000,91.369175,0.000000,0.005493,0.639995,0.288848,0.005819,0.000000,5.225000
50%,4.846200e+04,15.120000,46.350000,26.272040,39.517000,14.134000,39.929000,2021.0,0.410300,96.700000,0.000000,0.015654,0.831072,0.413624,0.015829,0.000000,6.466667
75%,6.040000e+06,30.155000,59.791665,29.785195,89.046000,29.148000,73.333000,2021.0,0.576300,98.722260,0.000000,0.069264,0.915424,0.572457,0.042217,0.000000,7.600000
max,8.309910e+11,288.980000,94.066670,104.569300,100.000000,100.000000,100.000000,2021.0,0.976000,100.000000,1.000000,0.990637,1.000000,1.000000,0.356940,1.000000,18.075000


How can case rate be 104%??

In [9]:
covid_df['fiftyplus'] = (covid_df['inperson_percent'] >= 50).astype(int)

In [14]:
pass_percent_fiftyplus = covid_df['pass_percent'].loc[covid_df['fiftyplus'] == 1].mean()
print(pass_percent_fiftyplus)

53.536901517857146


In [15]:
pass_percent_fiftyplus = covid_df['pass_percent'].loc[covid_df['fiftyplus'] == 0].mean()
print(pass_percent_fiftyplus)

40.885499335301056


In [16]:
# regression in 1(a)
fit = pf.feols('pass_percent ~ fiftyplus', data=covid_df, vcov='hetero')
fit.summary()

###

Estimation:  OLS
Dep. var.: pass_percent
sample: None = all
Inference:  hetero
Observations:  1519

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Intercept     |     40.885 |        0.572 |    71.516 |      0.000 | 39.764 |  42.007 |
| fiftyplus     |     12.651 |        0.920 |    13.749 |      0.000 | 10.846 |  14.456 |
---
RMSE: 17.564 R2: 0.113 


In [17]:
# regression in 1(b)
fit = pf.feols('pass_percent ~ fiftyplus + enrolment', data=covid_df, vcov='hetero')
fit.summary()

###

Estimation:  OLS
Dep. var.: pass_percent
sample: None = all
Inference:  hetero
Observations:  1519

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Intercept     |     40.535 |        0.718 |    56.477 |      0.000 | 39.127 |  41.943 |
| fiftyplus     |     12.822 |        0.951 |    13.481 |      0.000 | 10.956 |  14.687 |
| enrolment     |      0.010 |        0.014 |     0.741 |      0.459 | -0.017 |   0.037 |
---
RMSE: 17.56 R2: 0.114 


In [18]:
# part 1(c)
fit = pf.feols('pass_percent ~ case_rate + I(case_rate**2) + enrolment', data=covid_df, vcov='hetero')
fit.summary()

###

Estimation:  OLS
Dep. var.: pass_percent
sample: None = all
Inference:  hetero
Observations:  1519

| Coefficient       |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Intercept         |     30.101 |        3.373 |     8.925 |      0.000 | 23.485 |  36.717 |
| case_rate         |      1.004 |        0.203 |     4.941 |      0.000 |  0.605 |   1.402 |
| I(case_rate ** 2) |     -0.013 |        0.003 |    -4.044 |      0.000 | -0.019 |  -0.007 |
| enrolment         |     -0.027 |        0.013 |    -1.992 |      0.047 | -0.053 |  -0.000 |
---
RMSE: 18.455 R2: 0.021 


In [22]:
covid_df['case_rate_bin'] = pd.cut(
    x=covid_df['case_rate'],
    bins=[0, 10, 40, 80, np.inf],
    labels=['0-10', '10-40', '40-80', '80+'],
    right=False
)

In [23]:
# exclude 40-80 for dummy
fit = pf.feols('pass_percent ~ C(case_rate_bin, contr.treatment(base="40-80")) + enrolment', data=covid_df, vcov='hetero')
fit.summary()

###

Estimation:  OLS
Dep. var.: pass_percent
sample: None = all
Inference:  hetero
Observations:  1519

| Coefficient                                              |   Estimate |   Std. Error |   t value |   Pr(>|t|) |    2.5% |   97.5% |
|:---------------------------------------------------------|-----------:|-------------:|----------:|-----------:|--------:|--------:|
| Intercept                                                |     40.510 |        2.310 |    17.534 |      0.000 |  35.978 |  45.041 |
| C(case_rate_bin, contr.treatment(base='40-80'))[T.0-10]  |    -13.724 |        3.845 |    -3.570 |      0.000 | -21.265 |  -6.183 |
| C(case_rate_bin, contr.treatment(base='40-80'))[T.10-40] |      7.200 |        2.356 |     3.056 |      0.002 |   2.579 |  11.822 |
| C(case_rate_bin, contr.treatment(base='40-80'))[T.80+]   |     -8.020 |       13.599 |    -0.590 |      0.555 | -34.694 |  18.655 |
| enrolment                                                |     -0.033 |        0.013 |   

In [24]:
# exclude 80+ for dummy
fit = pf.feols('pass_percent ~ C(case_rate_bin, contr.treatment(base="80+")) + enrolment', data=covid_df, vcov='hetero')
fit.summary()

###

Estimation:  OLS
Dep. var.: pass_percent
sample: None = all
Inference:  hetero
Observations:  1519

| Coefficient                                            |   Estimate |   Std. Error |   t value |   Pr(>|t|) |    2.5% |   97.5% |
|:-------------------------------------------------------|-----------:|-------------:|----------:|-----------:|--------:|--------:|
| Intercept                                              |     32.490 |       13.404 |     2.424 |      0.015 |   6.197 |  58.783 |
| C(case_rate_bin, contr.treatment(base='80+'))[T.0-10]  |     -5.704 |       13.752 |    -0.415 |      0.678 | -32.679 |  21.270 |
| C(case_rate_bin, contr.treatment(base='80+'))[T.10-40] |     15.220 |       13.412 |     1.135 |      0.257 | -11.088 |  41.528 |
| C(case_rate_bin, contr.treatment(base='80+'))[T.40-80] |      8.020 |       13.599 |     0.590 |      0.555 | -18.655 |  34.694 |
| enrolment                                              |     -0.033 |        0.013 |    -2.486 |     

In [28]:
# investigate which bin we should have dropped
covid_df['case_rate_bin'].value_counts()

case_rate_bin
10-40    1482
0-10       18
40-80      16
80+         3
Name: count, dtype: int64

In [29]:
# 1(d)
fit = pf.feols('pass_percent ~ virtual_percent + case_rate + I(virtual_percent*case_rate)', data=covid_df, vcov='hetero')
fit.summary()

###

Estimation:  OLS
Dep. var.: pass_percent
sample: None = all
Inference:  hetero
Observations:  716

| Coefficient                    |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:-------------------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Intercept                      |     53.953 |        4.744 |    11.373 |      0.000 | 44.639 |  63.266 |
| virtual_percent                |     -0.282 |        0.109 |    -2.581 |      0.010 | -0.497 |  -0.067 |
| case_rate                      |     -0.127 |        0.168 |    -0.756 |      0.450 | -0.456 |   0.203 |
| I(virtual_percent * case_rate) |     -0.003 |        0.004 |    -0.717 |      0.473 | -0.011 |   0.005 |
---
RMSE: 17.048 R2: 0.178 


In [32]:
R = np.array(
    [[0, 1, 0, 0],
    [0, 0, 0, 1]]
)

q = np.array([0, 0])

fit.wald_test(R=R, q=q, distribution='F')

/Users/W.Zhouhan/GitHub/EC2C3/.venv/lib/python3.14/site-packages/pyfixest/estimation/models/feols_.py:1248: UserWarning: Distribution changed to chi2, as R is not an identity matrix and q is not a zero vector.
  warnings.warn(


statistic    2.324903e+02
pvalue       3.276231e-51
dtype: float64

In [34]:
R = np.array(
    [[0, 0, 1, 50]]
)

q = np.array([0])

fit.wald_test(R=R, q=q)

statistic    3.912059
pvalue       0.047941
dtype: float64